In [ ]:
    ############    #############   Pagination   #############   ##############   

 =>  Never return an entire table in one response -- both offset-based ('page 3, 20 per
       page') and cursor-based ('give me everything after id X') pagination exist; cursor
       is the better default for anything that changes while being paginated (offset
       pagination can skip/repeat rows if data is inserted/deleted mid-scroll).

 =>  A predictable list API always returns: the page of items, whether there's a next page,
       and how to fetch it (a cursor or an offset) -- never just a bare list.


In [ ]:
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel

ALL_ITEMS = [{"id": i, "name": f"item-{i}"} for i in range(1, 101)]

class Page(BaseModel):
    items: list[dict]
    next_cursor: int | None

app = FastAPI()

@app.get("/items", response_model=Page)
def list_items(cursor: int = Query(default=0), limit: int = Query(default=10, le=50)) -> Page:
    page_items = [i for i in ALL_ITEMS if i["id"] > cursor][:limit]
    next_cursor = page_items[-1]["id"] if len(page_items) == limit else None
    return Page(items=page_items, next_cursor=next_cursor)

client = TestClient(app)
page1 = client.get("/items", params={"limit": 5}).json()
print("page 1:", page1)
page2 = client.get("/items", params={"cursor": page1["next_cursor"], "limit": 5}).json()
print("page 2:", page2)


In [ ]:
 =>  The client never guesses an offset -- it just passes back whatever next_cursor it
       was given, which is robust even if items are added/removed between requests.

 =>  'limit: int = Query(default=10, le=50)' caps the page size server-side -- without this,
       a client could request limit=1000000 and force a huge, slow query.


In [ ]:
    ############    #############   Idempotency   #############   ##############   

 =>  An idempotent operation produces the same result no matter how many times it's
       repeated -- GET/PUT/DELETE are naturally idempotent by convention; POST (create) is
       NOT, which is exactly the problem when a client retries a timed-out 'create payment'
       request.

 =>  Fix: the client sends an Idempotency-Key header (a UUID it generates once per logical
       operation); the server remembers keys it has already processed and returns the SAME
       stored result instead of creating a duplicate.


In [ ]:
from fastapi import FastAPI, Header
from fastapi.testclient import TestClient
import uuid

app = FastAPI()
processed_requests: dict[str, dict] = {}
orders_created = 0

@app.post("/orders")
def create_order(idempotency_key: str = Header(...)):
    global orders_created
    if idempotency_key in processed_requests:
        return processed_requests[idempotency_key]  # same result, no duplicate side effect
    orders_created += 1
    result = {"order_id": orders_created, "status": "created"}
    processed_requests[idempotency_key] = result
    return result

client = TestClient(app)
key = str(uuid.uuid4())
print(client.post("/orders", headers={"idempotency-key": key}).json())
print(client.post("/orders", headers={"idempotency-key": key}).json())  # retry, same key
print("total orders actually created:", orders_created)


In [ ]:
 =>  Both calls return order_id 1 -- the retry (simulating a client that didn't see the
       first response and tried again) did NOT create a second order.

 =>  In production, processed_requests would be a table/cache with a TTL (e.g. Redis, or a
       Postgres table), not an in-memory dict -- it must survive across requests/restarts.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Add cursor-based pagination to a real endpoint backed by a database (order by a
           unique, indexed column, filter by 'id > cursor').

 =>  [ ] Store idempotency keys in Redis with a TTL (e.g. 24h) instead of an in-memory dict,
           so it survives a server restart.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Offset-based pagination ('LIMIT 20 OFFSET 40') on a frequently-changing table -- rows
       inserted/deleted during pagination can cause skipped or duplicated results.

 =>  Treating the idempotency key as optional -- if the client can omit it, a retry after a
       timeout goes right back to creating a duplicate.
